# WattWise — Random Forest Classifier (ISO 14955-5)

Notebook Google Colab untuk melatih, mengevaluasi, dan mengekspor model **Random Forest** klasifikasi status mesin mebel (`OFF`, `START_UP`, `IDLE`, `WORKING`) ke format **`rf-model.json`** yang langsung dibaca oleh aplikasi Next.js (`src/lib/ml/classifier.ts`).

### Dukungan Format Dataset
- **Mode A (Default — Simulasi):** Membuat dataset time-series profil daya 6 mesin furnitur.
- **Mode B (Google Sheets URL):** Fetch dataset langsung dari link Google Sheets publik (`Anyone with the link — Viewer` atau `Publish to web`).
- **Mode C (Upload CSV Lokal):** Mendukung file CSV 3 kolom dasar (`index`, `timestamp`, `daya` dalam Watt) + kolom `status` untuk training supervised.

> **Catatan Penting:** Kolom `index` hanya dipakai untuk mengurutkan baris dan **tidak dimasukkan** sebagai fitur ML agar model tidak overfit.

In [ ]:
import json
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, f1_score
from sklearn.model_selection import train_test_split

CLASSES = ["OFF", "START_UP", "IDLE", "WORKING"]
FEATURE_NAMES = [
    "power_ratio",
    "prev_power_ratio",
    "delta_power_ratio",
    "slope_per_sec",
    "rolling_mean_ratio",
    "rolling_std_ratio",
    "rolling_range_ratio",
]

print("Environment siap! Classes:", CLASSES)

## 1. Sumber Data (Google Sheets URL, Upload CSV, atau Simulasi)

In [ ]:
import re
from urllib.parse import urlparse, parse_qs, quote

# Opsi 1 (Disarankan): Tempel link Google Sheets publik ('Anyone with the link -> Viewer')
SPREADSHEET_URL = ""  # contoh: "https://docs.google.com/spreadsheets/d/1AbCd.../edit#gid=0"
SHEET_NAME = ""       # opsional: isi nama tab jika tidak memakai gid

# Opsi 2: Ubah ke True jika ingin upload file CSV lokal di Colab
USE_UPLOADED_CSV = False

# Kapasitas daya nominal mesin (Watt) jika spreadsheet/CSV tidak punya kolom `rated_power_w`
DEFAULT_RATED_POWER_W = 3800.0

def resolve_google_sheet_csv_url(url, sheet_name=""):
    parsed = urlparse(url.strip())
    if parsed.netloc == "docs.google.com" and "/spreadsheets/" in parsed.path:
        pub_match = re.search(r"/spreadsheets/d/e/([^/]+)", parsed.path)
        if pub_match:
            gid = parse_qs(parsed.query).get("gid", [None])[0]
            gid_part = f"gid={gid}&" if gid else ""
            return f"https://docs.google.com/spreadsheets/d/e/{pub_match.group(1)}/pub?{gid_part}single=true&output=csv"
        std_match = re.search(r"/spreadsheets/d/([a-zA-Z0-9-_]+)", parsed.path)
        if std_match:
            sheet_id = std_match.group(1)
            if sheet_name.strip():
                return f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={quote(sheet_name.strip())}"
            gid = parse_qs(parsed.query).get("gid", [None])[0]
            if not gid and parsed.fragment:
                m_gid = re.search(r"gid=(\d+)", parsed.fragment)
                gid = m_gid.group(1) if m_gid else None
            return f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv" + (f"&gid={gid}" if gid else "")
    return url.strip()

def normalize_dataset_df(raw_df, default_rated_w=DEFAULT_RATED_POWER_W):
    df = raw_df.copy()
    df.columns = [re.sub(r"[\s-]+", "_", str(c).strip().lower()) for c in df.columns]
    for alias in ["power_w", "power", "watt", "daya_w", "daya_aktif"]:
        if alias in df.columns and "daya" not in df.columns:
            df["daya"] = df[alias]
    for alias in ["recorded_at", "waktu", "time", "datetime", "tanggal"]:
        if alias in df.columns and "timestamp" not in df.columns:
            df["timestamp"] = df[alias]
    for alias in ["label", "state", "kelas", "class", "machine_status"]:
        if alias in df.columns and "status" not in df.columns:
            df["status"] = df[alias]
    if "index" not in df.columns:
        df["index"] = np.arange(1, len(df) + 1)
    if "machine_code" not in df.columns:
        df["machine_code"] = df["kode_mesin"] if "kode_mesin" in df.columns else "MESIN-01"
    if "rated_power_w" not in df.columns:
        df["rated_power_w"] = df["kapasitas_w"] if "kapasitas_w" in df.columns else default_rated_w
    df["daya"] = pd.to_numeric(df["daya"].astype(str).str.replace(",", ".", regex=False), errors="coerce")
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    if "status" in df.columns:
        status_map = {"MATI": "OFF", "STARTUP": "START_UP", "START-UP": "START_UP", "SIAGA": "IDLE", "STANDBY": "IDLE", "BEKERJA": "WORKING", "AKTIF": "WORKING", "PRODUKSI": "WORKING"}
        df["status"] = df["status"].astype(str).str.strip().str.upper().str.replace(" ", "_").replace(status_map)
        df = df[df["status"].isin(CLASSES)]
    return df.dropna(subset=["daya", "timestamp"]).reset_index(drop=True)

def generate_synthetic_furniture_df(seed=20261126):
    rng = np.random.default_rng(seed)
    profiles = [
        ("CNC-1325", 5500.0, 0.30, 0.81, 1.18),
        ("EDGE-01", 4200.0, 0.35, 0.80, 1.15),
        ("PSAW-01", 3800.0, 0.34, 0.82, 1.20),
        ("DUST-01", 2200.0, 0.40, 0.84, 1.16),
        ("BOR-01", 1800.0, 0.27, 0.78, 1.14),
        ("COMP-5HP", 3700.0, 0.29, 0.86, 1.22),
    ]
    records = []
    for machine_code, rated_w, idle_r, work_r, start_r in profiles:
        ts = pd.Timestamp("2026-11-26T08:00:00Z")
        idx = 1
        for _ in range(15):
            # OFF
            for _ in range(int(rng.integers(10, 15))):
                p = max(0.0, rng.normal(0.004, 0.003) * rated_w)
                records.append((idx, ts, round(p, 1), machine_code, rated_w, "OFF"))
                idx += 1; ts += pd.Timedelta(seconds=2)
            # START_UP
            for s in range(int(rng.integers(3, 6))):
                ratio = rng.normal(start_r * (1 - s * 0.08), 0.06)
                records.append((idx, ts, round(max(0.0, ratio * rated_w), 1), machine_code, rated_w, "START_UP"))
                idx += 1; ts += pd.Timedelta(seconds=2)
            # IDLE awal
            for _ in range(int(rng.integers(10, 15))):
                ratio = rng.normal(idle_r, 0.015)
                records.append((idx, ts, round(max(0.0, ratio * rated_w), 1), machine_code, rated_w, "IDLE"))
                idx += 1; ts += pd.Timedelta(seconds=2)
            # WORKING (fluktuasi beban pemotongan kayu)
            for w in range(int(rng.integers(16, 23))):
                wave = np.sin(w * 1.15) * 0.09
                ratio = np.clip(rng.normal(work_r + wave, 0.065), 0.50, 1.04)
                records.append((idx, ts, round(ratio * rated_w, 1), machine_code, rated_w, "WORKING"))
                idx += 1; ts += pd.Timedelta(seconds=2)
            # IDLE akhir
            for _ in range(int(rng.integers(10, 15))):
                ratio = rng.normal(idle_r, 0.016)
                records.append((idx, ts, round(max(0.0, ratio * rated_w), 1), machine_code, rated_w, "IDLE"))
                idx += 1; ts += pd.Timedelta(seconds=2)
    return pd.DataFrame(records, columns=["index", "timestamp", "daya", "machine_code", "rated_power_w", "status"])

if SPREADSHEET_URL.strip():
    csv_url = resolve_google_sheet_csv_url(SPREADSHEET_URL, SHEET_NAME)
    print("Mengambil dataset dari:", csv_url)
    df = normalize_dataset_df(pd.read_csv(csv_url))
elif USE_UPLOADED_CSV:
    from google.colab import files
    uploaded = files.upload()
    csv_name = next(iter(uploaded.keys()))
    df = normalize_dataset_df(pd.read_csv(csv_name))
else:
    df = generate_synthetic_furniture_df()

print(f"Total baris dataset: {len(df)}")
display(df.head(10))
print("\nDistribusi kelas:")
print(df["status"].value_counts())

## 2. Visualisasi Profil Daya & Status ISO 14955-5

In [ ]:
sample_machine = df["machine_code"].iloc[0]
sub = df[df["machine_code"] == sample_machine].head(120)

palette = {"OFF": "#64748b", "START_UP": "#f97316", "IDLE": "#f59e0b", "WORKING": "#10b981"}
plt.figure(figsize=(13, 4))
plt.plot(sub["index"], sub["daya"], color="#94a3b8", linewidth=1, alpha=0.6)
for st, grp in sub.groupby("status"):
    plt.scatter(grp["index"], grp["daya"], label=st, color=palette.get(st, "#fff"), s=24)
plt.title(f"Profil Daya Time-Series ({sample_machine}) — Standar ISO 14955-5")
plt.xlabel("Sample Index")
plt.ylabel("Daya Aktif (Watt)")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 3. Feature Engineering (Identik dengan `src/lib/ml/features.ts`)
Menghasilkan 7 fitur *sliding window* (5 sampel terakhir) hanya dari **`timestamp` + `daya (Watt)`** yang dinormalisasi terhadap `rated_power_w`.

In [ ]:
def extract_features_df(raw_df, window_size=5):
    feat_rows = []
    labels = []

    for _, grp in raw_df.groupby("machine_code", sort=False):
        grp = grp.sort_values(["timestamp", "index"] if "index" in grp.columns else ["timestamp"]).reset_index(drop=True)
        powers = grp["daya"].clip(lower=0).to_numpy(dtype=float)
        rated = np.maximum(100.0, grp["rated_power_w"].to_numpy(dtype=float))
        ts_sec = grp["timestamp"].astype("int64").to_numpy() / 1e9
        has_status = "status" in grp.columns
        statuses = grp["status"].to_numpy() if has_status else None

        for i in range(2, len(grp)):
            start = max(0, i - window_size + 1)
            win_p = powers[start : i + 1]
            safe_r = rated[i]
            ratios = win_p / safe_r

            power_ratio = powers[i] / safe_r
            prev_power_ratio = powers[i - 1] / safe_r
            delta_power_ratio = power_ratio - prev_power_ratio

            raw_dt = ts_sec[i] - ts_sec[i - 1]
            dt_sec = min(raw_dt, 60.0) if raw_dt > 0 else 2.0
            slope_per_sec = delta_power_ratio / dt_sec

            rolling_mean_ratio = float(np.mean(ratios))
            rolling_std_ratio = float(np.std(ratios, ddof=0))
            rolling_range_ratio = float(np.max(ratios) - np.min(ratios))

            feat_rows.append([
                round(power_ratio, 6),
                round(prev_power_ratio, 6),
                round(delta_power_ratio, 6),
                round(slope_per_sec, 6),
                round(rolling_mean_ratio, 6),
                round(rolling_std_ratio, 6),
                round(rolling_range_ratio, 6),
            ])
            if has_status:
                labels.append(statuses[i])

    X = pd.DataFrame(feat_rows, columns=FEATURE_NAMES)
    y = pd.Series(labels, name="status") if labels else None
    return X, y

X, y = extract_features_df(df)
print("Shape fitur:", X.shape)
display(X.head())

## 4. Training & Evaluasi `RandomForestClassifier` (scikit-learn)

In [ ]:
y_idx = y.map({cls: i for i, cls in enumerate(CLASSES)})
X_train, X_test, y_train, y_test = train_test_split(
    X, y_idx, test_size=0.2, random_state=42, stratify=y_idx
)

rf = RandomForestClassifier(
    n_estimators=25,
    max_depth=7,
    min_samples_leaf=4,
    max_features=3,
    random_state=42,
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {acc * 100:.2f}%\n")
print(classification_report(y_test, y_pred, target_names=CLASSES, digits=4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Confusion Matrix (ISO 14955-5)")

imp = pd.Series(rf.feature_importances_, index=FEATURE_NAMES).sort_values()
imp.plot(kind="barh", ax=axes[1], color="#f97316")
axes[1].set_title("Feature Importance — Random Forest")
axes[1].set_xlabel("Importance")
plt.tight_layout()
plt.show()

## 5. Export Model ke `rf-model.json` untuk Next.js (`src/lib/ml/rf-model.json`)
Format JSON yang dihasilkan 100% kompatibel dengan `src/lib/ml/classifier.ts`.

In [ ]:
def export_sklearn_rf_to_wattwise_json(model, test_acc, n_train, n_test, f1_per_class):
    trees = []
    for estimator in model.estimators_:
        t = estimator.tree_
        nodes = []
        for i in range(t.node_count):
            is_leaf = t.children_left[i] == t.children_right[i]
            counts = t.value[i][0]
            total = float(np.sum(counts)) or 1.0
            probs = [round(float(c / total), 4) for c in counts]
            nodes.append({
                "featureIndex": -1 if is_leaf else int(t.feature[i]),
                "threshold": 0.0 if is_leaf else round(float(t.threshold[i]), 6),
                "left": -1 if is_leaf else int(t.children_left[i]),
                "right": -1 if is_leaf else int(t.children_right[i]),
                "probs": probs,
            })
        trees.append(nodes)

    return {
        "version": "1.0.0",
        "modelType": "RandomForestClassifier",
        "standard": "ISO 14955-5:2020",
        "trainedAt": datetime.now(timezone.utc).isoformat(),
        "trainedWith": "google-colab-scikit-learn",
        "classes": CLASSES,
        "featureNames": FEATURE_NAMES,
        "nEstimators": len(trees),
        "maxDepth": int(model.max_depth or 0),
        "metrics": {
            "testAccuracy": round(float(test_acc), 4),
            "trainSamples": int(n_train),
            "testSamples": int(n_test),
            "perClassF1": f1_per_class,
        },
        "trees": trees,
    }

f1_scores = f1_score(y_test, y_pred, average=None)
f1_map = {CLASSES[i]: round(float(f1_scores[i]), 4) for i in range(len(CLASSES))}

artifact = export_sklearn_rf_to_wattwise_json(rf, acc, len(X_train), len(X_test), f1_map)
with open("rf-model.json", "w", encoding="utf-8") as f:
    json.dump(artifact, f)

import os
print(f"Berhasil ekspor rf-model.json ({os.path.getsize('rf-model.json') / 1024:.1f} KB)")

# Otomatis unduh jika dijalankan di Google Colab
try:
    from google.colab import files
    files.download("rf-model.json")
except ImportError:
    print("Jalankan di Google Colab untuk auto-download, atau salin rf-model.json ke src/lib/ml/rf-model.json")